# Prepare sciPlex3 + Prophet embeddings

Single-dataset version of `prepare_tahoe_prophet`. Adds a `prophet_emb` dict
(name -> 512d) to `uns` and filters cells to control + prophet-matched drugs.

**Run the *Inspect* cell first** and check the three schema variables
(`SOURCE_DRUG_COL`, `CELL_LINE_COL`, `DRUG_EMB_UNS`) match this file — they are
set to the Tahoe `_w_emb` convention and may differ for sciPlex.

In [1]:
import gc
import h5py
import numpy as np
import pandas as pd
import anndata as ad
import torch
from pathlib import Path
from prophet import Prophet
from huggingface_hub import hf_hub_download
from prophet.data.dataloader import read_in_priors
from tqdm import tqdm

SCIPLEX_H5AD = Path("/storage/pancellflow/data/srivatsan20_sciplex3_w_emb.h5ad")
OUTPUT_H5AD  = Path("/storage/pancellflow/filtered_data/sciplex3_prophet_filtered.h5ad")

# ── schema (verify with the Inspect cell; adjust if sciPlex differs) ──────────
SOURCE_DRUG_COL = "drug_0"               # obs column holding the drug name
CELL_LINE_COL   = "cell_line"            # obs column holding the cell line
DRUG_EMB_UNS    = "drug_0_embeddings"    # uns key with the per-drug embedding dict
CELL_EMB_UNS    = "cell_line_embeddings" # uns key with the per-cell-line embedding dict
CONTROL_LABEL   = "control"              # value in SOURCE_DRUG_COL marking controls

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.serialization.add_safe_globals([np.core.multiarray.scalar])
torch.serialization.safe_globals([np.core.multiarray.scalar])

## Inspect

Confirm the schema variables above before going further.

In [3]:
with h5py.File(SCIPLEX_H5AD, "r") as f:
    obs_cols = list(f["obs"].keys())
    uns_keys = list(f["uns"].keys())
    obsm_keys = list(f["obsm"].keys())

print("obs columns :", obs_cols)
print("uns keys    :", uns_keys)
print("obsm keys   :", obsm_keys)

assert SOURCE_DRUG_COL in obs_cols, f"{SOURCE_DRUG_COL!r} not in obs — pick the drug column above"
assert CELL_LINE_COL in obs_cols,   f"{CELL_LINE_COL!r} not in obs — pick the cell-line column above"
assert DRUG_EMB_UNS in uns_keys,    f"{DRUG_EMB_UNS!r} not in uns — pick the drug-embedding key above"

# peek at the drug column values
obs_drug = ad.io.read_elem(h5py.File(SCIPLEX_H5AD, "r")["obs"])[SOURCE_DRUG_COL]
print("\nn drug values :", obs_drug.nunique())
print("control present:", (obs_drug == CONTROL_LABEL).any())
print("sample drugs   :", [d for d in obs_drug.unique()[:10]])

obs columns : ['_index', 'assay', 'cancer', 'cell_line', 'cell_type', 'development_stage', 'disease', 'donor_id', 'dose_unit', 'dose_value', 'drug_0', 'drug_1', 'drug_org', 'ncounts', 'organism', 'pathway', 'pathway_level_1', 'pathway_level_2', 'pert_compound', 'pert_dose', 'pert_name', 'pert_target', 'pert_time', 'pert_type', 'plate', 'replicate', 'self_reported_ethnicity', 'sex', 'suspension_type', 'time', 'tissue', 'tissue_type', 'well']
uns keys    : ['cell_line_ccle_embedding_dim', 'cell_line_ccle_embeddings', 'cell_line_embedding_dim', 'cell_line_embeddings', 'drug_0_embeddings', 'drug_1_embeddings', 'drug_embedding_dim']
obsm keys   : ['X_scconcept', 'X_scgpt', 'X_scimilarity', 'X_scimilarity_correct', 'X_state']

n drug values : 181
control present: True
sample drugs   : ['TAK-901', 'AG-490', 'abexinostat', 'alisertib', 'busulfan', 'Obatoclax', 'Enzastaurin', 'BMS-265246', 'UNC0379', 'Raltitrexed']


## Drugs present in sciPlex

In [4]:
adata_obs = ad.io.read_elem(h5py.File(SCIPLEX_H5AD, "r")["obs"])
sciplex_drugs = [d for d in adata_obs[SOURCE_DRUG_COL].unique() if d != CONTROL_LABEL]
del adata_obs
gc.collect()
print(f"sciPlex drugs: {len(sciplex_drugs)}")

sciPlex drugs: 180


## Load Prophet model

In [5]:
import torch

_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs["weights_only"] = False          # torch 2.6 flipped this default to True
    return _orig_load(*args, **kwargs)
torch.load = _patched_load

In [6]:
model = Prophet.from_pretrained("base", split="cell_lines", fold=0, seed=110)
model.model.eval()

🔄 Downloading base model from HuggingFace Hub...
   Configuration: split=cell_lines, fold=0, seed=110
Successfully downloaded base model: epoch=29-step=45360.ckpt
  Configuration: split=unseen_cell_lines, fold=0, seed=110
  Model file: /root/.cache/prophet/base_cell_lines_fold0_seed110/datasets--theislab--Prophet/snapshots/7d8b27edeaab117af851e3f5625a018b0f45fa35/base_pretrained/unseen_cell_lines_fold_0/seed_110/epoch=29-step=45360.ckpt
Learning rate set to 1e-05


TransformerPredictor(
  (learnable_embedding): Embedding(1000, 512, max_norm=0.5)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (gene_net): Sequential(
    (0): Linear(in_features=1219, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
  (drug_net): Sequential(
    (0): Linear(in_features=1219, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
  (cl_net): Sequential(
    (0): Linear(in_features=300, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-7): 8 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQu

## IV embeddings — match drugs by name

In [7]:
files = [
    "embeddings/intervention_embeddings/global_iv_scaledv3.csv",
]
paths = [hf_hub_download(repo_id="theislab/Prophet", filename=f, repo_type="dataset") for f in files]
iv_emb1 = read_in_priors(paths)

iv_drugs1 = iv_emb1[iv_emb1["type"] == "drug"].copy()
iv_drugs1 = iv_drugs1[~iv_drugs1.index.duplicated(keep="first")]
iv_name_lookup = {n.strip().lower(): n for n in iv_drugs1.index}

emb_cols = [c for c in iv_drugs1.columns if c not in ("type", "smiles")]
iv_vectors = iv_drugs1[emb_cols]

In [8]:
matched_names, matched_vecs = [], []
for drug in sciplex_drugs:
    drug_norm = str(drug).strip().lower()
    if drug_norm in iv_name_lookup:
        idx = iv_name_lookup[drug_norm]
        matched_names.append(drug)
        matched_vecs.append(iv_vectors.loc[idx].values.astype(np.float32))

print(f"Matched: {len(matched_names)} / {len(sciplex_drugs)} drugs")

Matched: 128 / 180 drugs


## Prophet embeddings for matched drugs

In [9]:
with torch.no_grad():
    embs = model.model.drug_net(
        torch.tensor(np.stack(matched_vecs), dtype=torch.float32)
    ).numpy()

prophet_emb = {name: embs[i] for i, name in enumerate(matched_names)}
print(f"prophet_emb: {len(prophet_emb)} drugs x {embs.shape[1]}d")

del model, iv_emb1, iv_drugs1
gc.collect()

prophet_emb: 128 drugs x 512d


0

## Filter + write

Single file: keep control + prophet-matched drugs, attach `prophet_emb`, write.

In [10]:
prophet_drug_set = set(prophet_emb.keys())

with h5py.File(SCIPLEX_H5AD, "r") as f:
    a = ad.AnnData(
        X=ad.io.read_elem(f["X"]) if "X" in f else None,
        obs=ad.io.read_elem(f["obs"]),
        obsm=ad.io.read_elem(f["obsm"]),
        uns=ad.io.read_elem(f["uns"]),
    )
    
# keep: real cell line  AND  (control OR prophet-matched drug)
keep = (
    a.obs[CELL_LINE_COL].notna()
    & (a.obs[CELL_LINE_COL].astype(str) != "NA")
    & ((a.obs[SOURCE_DRUG_COL] == CONTROL_LABEL) | a.obs[SOURCE_DRUG_COL].isin(prophet_drug_set))
)
print(f"kept {int(keep.sum()):,} / {len(keep):,} cells")
a = a[keep].copy()

# drop now-empty 'NA' category so it doesn't become an empty distribution group
if a.obs[CELL_LINE_COL].dtype.name == "category":
    a.obs[CELL_LINE_COL] = a.obs[CELL_LINE_COL].cat.remove_unused_categories()

# attach prophet embeddings (drug_0_embeddings / cell_line_embeddings already in uns)
a.uns["prophet_emb"] = prophet_emb

# CellFlow expects a bool 'control' and a 'drug' column
a.obs["control"] = (a.obs[SOURCE_DRUG_COL] == CONTROL_LABEL)
a.obs["drug"]    = a.obs[SOURCE_DRUG_COL].astype(str)
if SOURCE_DRUG_COL != "drug":
    a.obs.drop(columns=[SOURCE_DRUG_COL], inplace=True)

print(f"\nTotal cells : {a.n_obs:,}")
print(f"Genes       : {a.n_vars:,}")
print(f"cell lines  : {sorted(a.obs[CELL_LINE_COL].unique())}")
print(f"drugs (+ctrl): {a.obs['drug'].nunique()}")
print(f"uns keys    : {list(a.uns.keys())}")

a.write_h5ad(OUTPUT_H5AD)
print(f"Saved to {OUTPUT_H5AD}")

kept 518,123 / 769,186 cells

Total cells : 518,123
Genes       : 58,302
cell lines  : ['A549', 'K-562', 'MCF7']
drugs (+ctrl): 129
uns keys    : ['cell_line_ccle_embedding_dim', 'cell_line_ccle_embeddings', 'cell_line_embedding_dim', 'cell_line_embeddings', 'drug_0_embeddings', 'drug_1_embeddings', 'drug_embedding_dim', 'prophet_emb']
Saved to /storage/pancellflow/filtered_data/sciplex3_prophet_filtered.h5ad


## Verify

In [11]:
with h5py.File(OUTPUT_H5AD, "r") as f:
    print(list(f.keys()))
    print(list(f["uns"].keys()))
    print("prophet_emb drugs:", len(f["uns"]["prophet_emb"].keys()))

adata = ad.read_h5ad(OUTPUT_H5AD, backed="r")
print(adata)
print("prophet_emb sample:", list(adata.uns["prophet_emb"].keys())[:5])

['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']
['cell_line_ccle_embedding_dim', 'cell_line_ccle_embeddings', 'cell_line_embedding_dim', 'cell_line_embeddings', 'drug_0_embeddings', 'drug_1_embeddings', 'drug_embedding_dim', 'prophet_emb']
prophet_emb drugs: 128
AnnData object with n_obs × n_vars = 518123 × 58302 backed at '/storage/pancellflow/filtered_data/sciplex3_prophet_filtered.h5ad'
    obs: 'pert_target', 'pert_compound', 'pert_dose', 'pert_time', 'organism', 'cell_line', 'cell_type', 'disease', 'tissue_type', 'tissue', 'assay', 'suspension_type', 'donor_id', 'sex', 'self_reported_ethnicity', 'development_stage', 'pert_name', 'pert_type', 'ncounts', 'well', 'plate', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'pathway', 'dose_unit', 'cancer', 'drug_org', 'drug_1', 'control', 'drug'
    uns: 'cell_line_ccle_embedding_dim', 'cell_line_ccle_embeddings', 'cell_line_embedding_dim', 'cell_line_embeddings', 'drug_0_embeddings', 'drug_1

In [25]:
set(adata.obs["dose_value"].values)

{0.0, 10.0, 100.0, 1000.0, 10000.0}

In [17]:
adata.uns["drug_1_embeddings"].keys()

dict_keys(['control'])